# Orbit Wars — Colab DAgger Training Pipeline

GPU-accelerated RL training on Google Colab with persistent storage via Google Drive.

**Pipeline:**
1. Setup (install deps, mount Drive, clone repo)
2. Configure experiment (DAgger = BC from hybrid demos + PPO with imitation decay)
3. Train (phases: demo collection → BC pretrain → PPO)
4. Submit model to Kaggle
5. Evaluate against baselines
6. Monitor with TensorBoard

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

# ── Clone or update repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT THIS
REPO_DIR = '/content/orbit_wars'

if os.path.exists(REPO_DIR):
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# ── Install dependencies ────────────────────────────────────────────────────
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')
os.system('pip install -q "stable-baselines3[extra]>=2.3" gymnasium pyyaml tensorboard')

print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification

import torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected! Training will be slow.')
    print('Go to Runtime > Change runtime type > GPU')
    DEVICE = 'cpu'

print(f'\nUsing device: {DEVICE}')

## Configuration

Edit the cell below to configure your experiment. The DAgger config runs:
1. **Demo collection** — hybrid agent plays games, recording (state, action) pairs
2. **BC pretraining** — supervised learning on expert demonstrations
3. **PPO + imitation decay** — RL training with linearly decaying imitation loss
4. **Periodic evaluation** — win rate tracking against apex/random

In [ ]:
#@title 3. Experiment Config

from src.config import load_train_config

# Load DAgger config (BC pretrain + PPO with imitation decay)
cfg = load_train_config('configs/transformer_dagger.yaml')

# ── H100 Colab overrides ────────────────────────────────────────────────────
cfg.device = 'cuda'
cfg.ppo.num_envs = 4
cfg.ppo.rollout_steps = 128
cfg.ppo.total_updates = 5000

cfg.eval.eval_every = 250
cfg.eval.eval_games = 10

# Point outputs to Google Drive for persistence
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'

print(f'Run: {cfg.run_name}')
print(f'Device: {cfg.device}')
print(f'Updates: {cfg.ppo.total_updates}')
print(f'Envs: {cfg.ppo.num_envs}, rollout_steps: {cfg.ppo.rollout_steps}')
print(f'Opponent: {cfg.opponent}')
print(f'Imitation: enabled={cfg.imitation.enabled}, bc_games={cfg.imitation.bc_games}, '
      f'bc_epochs={cfg.imitation.bc_epochs}, coef_decay={cfg.imitation.coef_decay_updates}')
print(f'Eval: every {cfg.eval.eval_every} updates, {cfg.eval.eval_games} games vs {cfg.eval.eval_opponents}')

In [ ]:
#@title 4. Train

import yaml, sys, os
sys.path.insert(0, '/content/orbit_wars')
os.chdir('/content/orbit_wars')

# Write config to temp file for the training script
temp_cfg = '/tmp/train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump({
        'seed': cfg.seed,
        'run_name': cfg.run_name,
        'device': cfg.device,
        'save_dir': cfg.save_dir,
        'log_dir': cfg.log_dir,
        'checkpoint_every': cfg.checkpoint_every,
        'log_every': cfg.log_every,
        'opponent': cfg.opponent,
        'self_play_update_interval': cfg.self_play_update_interval,
        'self_play_deterministic': cfg.self_play_deterministic,
        'alternate_player_sides': cfg.alternate_player_sides,
        'env': {
            'max_targets': cfg.env.max_targets,
            'k_neighbors': cfg.env.k_neighbors,
            'ship_fractions': cfg.env.ship_fractions,
        },
        'model': {
            'embed_dim': cfg.model.embed_dim,
            'n_heads': cfg.model.n_heads,
            'n_layers': cfg.model.n_layers,
            'ff_dim': cfg.model.ff_dim,
            'pos_hidden': cfg.model.pos_hidden,
        },
        'ppo': {
            'rollout_steps': cfg.ppo.rollout_steps,
            'num_envs': cfg.ppo.num_envs,
            'total_updates': cfg.ppo.total_updates,
            'epochs': cfg.ppo.epochs,
            'minibatch_size': cfg.ppo.minibatch_size,
            'gamma': cfg.ppo.gamma,
            'clip_coef': cfg.ppo.clip_coef,
            'ent_coef': cfg.ppo.ent_coef,
            'vf_coef': cfg.ppo.vf_coef,
            'lr': cfg.ppo.lr,
            'max_grad_norm': cfg.ppo.max_grad_norm,
        },
        'reward': {
            'sparse': cfg.reward.sparse,
            'dense_ship_coef': cfg.reward.dense_ship_coef,
            'dense_prod_coef': cfg.reward.dense_prod_coef,
        },
        'imitation': {
            'enabled': cfg.imitation.enabled,
            'bc_games': cfg.imitation.bc_games,
            'bc_demo_opponent': cfg.imitation.bc_demo_opponent,
            'bc_epochs': cfg.imitation.bc_epochs,
            'bc_lr': cfg.imitation.bc_lr,
            'bc_batch_size': cfg.imitation.bc_batch_size,
            'coef_start': cfg.imitation.coef_start,
            'coef_decay_updates': cfg.imitation.coef_decay_updates,
            'distilled_opponent': cfg.imitation.distilled_opponent,
        },
        'eval': {
            'eval_every': cfg.eval.eval_every,
            'eval_games': cfg.eval.eval_games,
            'eval_opponents': cfg.eval.eval_opponents,
        },
    }, f)

!python -m src.train --config {temp_cfg}

CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'
print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')

## Submission

Generate and submit to Kaggle. The apex agent is a reliable baseline submission.
The trained transformer checkpoint is saved to Drive for future submission generation.

In [ ]:
#@title 5. Generate Submission
import shutil
from pathlib import Path
from agents.rl_agent import export_submission

# ── Save apex submission (reliable baseline) ────────────────────────────────
submission_path = f'{DRIVE_OUTPUT}/submissions/submission_apex.py'
export_submission(None, output_path=submission_path, mode='apex')
print(f'Apex submission: {submission_path}')

# ── Save hybrid submission ──────────────────────────────────────────────────
hybrid_path = f'{DRIVE_OUTPUT}/submissions/submission_hybrid.py'
export_submission(None, output_path=hybrid_path, mode='hybrid')
print(f'Hybrid submission: {hybrid_path}')

# ── Copy trained checkpoint to submissions dir ──────────────────────────────
ckpt_src = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if ckpt_src.exists():
    ckpt_dst = f'{DRIVE_OUTPUT}/submissions/ckpt_last.pt'
    shutil.copy2(ckpt_src, ckpt_dst)
    print(f'Trained checkpoint copied to: {ckpt_dst}')
else:
    print(f'No checkpoint found at {ckpt_src}')

print(f'\nTo upload apex submission:')
print(f'  kaggle competitions submit orbit-wars -f "{submission_path}" -m "Apex agent"')
print(f'\nTo upload hybrid submission:')
print(f'  kaggle competitions submit orbit-wars -f "{hybrid_path}" -m "Hybrid agent"')

## Evaluation

Run the trained transformer policy head-to-head against apex and random agents.

In [ ]:
#@title 6. Evaluate Trained Model

import torch
from pathlib import Path
from src.config import load_train_config
from src.policy import TransformerPolicy
from src.logging import make_eval_agent
from evaluation.evaluate import run_games, print_results

# Load checkpoint
ckpt_path = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if not ckpt_path.exists():
    print(f'No checkpoint found at {ckpt_path}')
    print('Available checkpoints:')
    for p in sorted(Path(DRIVE_OUTPUT, 'checkpoints').rglob('ckpt_last.pt')):
        print(f'  {p}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    policy = TransformerPolicy(cfg.model, cfg.env).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    policy.load_state_dict(ckpt['policy'])
    policy.eval()
    print(f'Loaded checkpoint: {ckpt_path} (update {ckpt["update"]})')

    eval_agent = make_eval_agent(policy, cfg, device)

    N_GAMES = 20

    from agents.apex import agent as apex_agent
    print(f'\nRL vs Apex ({N_GAMES} games):')
    r = run_games(eval_agent, apex_agent, n_games=N_GAMES, verbose=True)
    print_results('rl_trained', 'apex', r)

    from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent
    print(f'\nRL vs Random ({N_GAMES} games):')
    r2 = run_games(eval_agent, random_agent, n_games=N_GAMES, verbose=True)
    print_results('rl_trained', 'random', r2)

## Monitoring

TensorBoard shows train/*, eval/*, and bc/* metrics.
**Note:** TensorBoard may block execution of cells below it — run this last.

In [ ]:
#@title 7. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs